In [1]:
print(123)

123


In [6]:
"""
Q1. Embedding a query
Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

-0.31
-0.02
0.12
0.44
"""

# fetch embedder

from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)

print(v1[0])

# SOLUTION: -0.02058203437252893

-0.02058203437252893


In [7]:
"""
Loading the data
We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
Each document is a dictionary with a filename and content, and there are 72 pages.
"""

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

"""
Q2. Cosine similarity
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

0.07
0.37
0.68
0.92
"""

'\nQ2. Cosine similarity\nThe embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.\n\nTake the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?\n\n0.07\n0.37\n0.68\n0.92\n'

In [11]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [13]:
content_to_embed = None
for d in documents:
    if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        content_to_embed = d['content']

# embed
vm2 = embed.encode(content_to_embed)


In [ ]:
# calculat cosine similarity

# compute similarities
v1.dot(vm2)

# SOLUTION: np.float64(0.36107027225589694)

np.float64(0.36107027225589694)

In [15]:
"""
Q3. Chunking and search by hand
A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

scores = X.dot(v)
Which file does the highest-scoring chunk belong to (its filename)?

02-vector-search/lessons/03-embeddings-dataset.md
02-vector-search/lessons/06-rag-vector.md
02-vector-search/lessons/07-sqlitesearch-vector.md
02-vector-search/lessons/09-onnx-embedder.md

"""


"\nQ3. Chunking and search by hand\nA full page covers several topics, which waters down its embedding.\n\nWe chunk the pages the same way as in homework 1:\n\nfrom gitsource import chunk_documents\nchunks = chunk_documents(documents, size=2000, step=1000)\nWe embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:\n\nscores = X.dot(v)\nWhich file does the highest-scoring chunk belong to (its filename)?\n\n02-vector-search/lessons/03-embeddings-dataset.md\n02-vector-search/lessons/06-rag-vector.md\n02-vector-search/lessons/07-sqlitesearch-vector.md\n02-vector-search/lessons/09-onnx-embedder.md\n\n"

In [21]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

chunks[:2]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [22]:
# contents 
contents = [
    chunk['content'] for chunk in chunks
]

contents[:2]

['# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language

In [ ]:
vectors = embed.encode_batch(
    contents
)

In [26]:
import numpy as np
X = np.array(vectors)

In [30]:
scores = X.dot(v1)
scores[2]

np.float64(0.05905590366174358)

In [ ]:
top5 = np.argsort(scores)[-5:]  # gives sorted values; then give top 5 largest 
top5 = top5[::-1]
top5



array([ 94,  14, 162,  85, 253])

In [33]:
# TRICK

 # instead of np.argsort(scores)[-5:]: 
top5 = np.argsort(-scores)[:5]
top5

array([ 94,  14, 162,  85, 253])

In [ ]:
for idx in top5:
    print(scores[idx])
    # print(documents[idx])
    print(chunks[idx]['filename'])
    print()

# --> SOLUTION: 02-vector-search/lessons/07-sqlitesearch-vector.md

0.6489017718578813
02-vector-search/lessons/07-sqlitesearch-vector.md

0.5510346348435939
01-agentic-rag/lessons/05-search.md

0.4065607072060337
04-evaluation/lessons/05-search-metrics.md

0.40618197902402514
02-vector-search/lessons/04-vector-search.md

0.40610594157880786
06-best-practices/lessons/03-reranking.md



In [38]:
"""
Q4. Vector search with minsearch
We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

02-vector-search/lessons/04-vector-search.md
04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/13-llm-as-judge.md
05-monitoring/lessons/04-metrics.md
"""

"\nQ4. Vector search with minsearch\nWe've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.\n\nLet's use VectorSearch from minsearch and run a search for the following query:\n\nWhat metric do we use to evaluate a search engine?\n\nWhich file is the filename of the first result?\n\n02-vector-search/lessons/04-vector-search.md\n04-evaluation/lessons/05-search-metrics.md\n04-evaluation/lessons/13-llm-as-judge.md\n05-monitoring/lessons/04-metrics.md\n"

In [39]:
# we just did all of these low level things manually but now will use a library

q4 = "What metric do we use to evaluate a search engine?"
v4 = embed.encode(q4)

from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])  # keyword_fields arg has the purpose of being able to do keywrod search with that later
vindex.fit(X, chunks)  # documents


In [ ]:
vindex.search(v4)[0] 

# --> SOLUTION: 'filename': '04-evaluation/lessons/05-search-metrics.md'

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

In [47]:
"""
Q5. Text search vs vector search
Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

02-vector-search/lessons/01-intro.md
02-vector-search/lessons/02-embeddings.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
"""

q5 = "How do I store vectors in PostgreSQL?"

In [51]:
# Index from minsearch 

from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

index = build_index(chunks)

search_result = index.search(
    q5,
)

index_search_top5 = search_result[:5]
index_search_top5

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [52]:
# VectorSearch from minsearch

v5 = embed.encode(q5)

vector_search_top5 = vindex.search(v5)[:5] 
vector_search_top5


[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO